# Extract extremes from a time series using Peak Over Threshold (POT)

This notebook performs the following steps:

1. **Load data**  
   Non-tidal residual (NTR) data are stored as `.pkl` files, each containing a `pandas.Series` with a datetime index and a column `'NTR'` representing residual level. The station metadata is provided in a `stations_info.csv` file, which contains station information such as name, longitude, and latitude.

2. **Use Peak Over Threshold (POT) method to extract events**  


## 1. Load data

In [1]:
import pandas as pd

# Load station info (names,lon,lat)
stations = pd.read_csv("inputs/stations_info.csv", usecols=["station", "lon", "lat"])
stations

,station,lon,lat
0,Huibertgat,6.398433,53.573748
1,Schiermonnikoog,6.202908,53.468935
2,Eemshaven,6.828392,53.448699
3,Nes,5.758907,53.429936
4,Lauwersoog,6.196952,53.408684
5,West_Terschelling,5.220026,53.363043
6,Vlieland_Haven,5.091456,53.296125
7,Harlingen,5.409342,53.175632
8,Oudeschild,4.850192,53.038833
9,Den_Helder,4.744990,52.964357


In [2]:
# Load non-tidal residual data per station in a list
NTR = []
for i in range(len(stations)):
    ntr_element = pd.read_pickle(f"inputs/NTR/ntr_{i}.pkl")
    # Append loaded data to lists
    NTR.append(ntr_element)

## 2. Use Peak Over Threshold (POT) Method to Extract Events

For additional options and details, see the function [`pot`](./extremes.py) in the `extremes.py` module.

In [3]:
from utils.extremes import pot

# Parameters
indep = 3 * 24  # set independence between events (hours)
res = 1  # minutes
resunit = "min"  # the lowest frequency in this data is 1 min
mindur = 6  # set minimum duration of event to keep (hours)
qopt = "quan"  # 'quan' (quantile) or 'thre' (set threshold)
eventopt = "exc"  # exc, gaps, cont (see function for details)
q = 0.70  # 70th percentile

extremes_NTR = []
events_NTR = []
threshold_NTR = []
for ntr_data in NTR:
    extremes_ntr, events_ntr, threshold_ntr = pot(
        ntr_data, indep, mindur, eventopt, res, resunit, qopt, q
    )
    # Append the data for the current station to the respective lists
    extremes_NTR.append(extremes_ntr)
    events_NTR.append(events_ntr)
    threshold_NTR.append(threshold_ntr)

# cleanup
del extremes_ntr, events_ntr, threshold_ntr, ntr_data

1973-01-01 07:00:00    0.106786
1973-01-01 10:00:00    0.152441
1973-01-15 13:00:00    0.106237
1973-01-22 14:00:00    0.166259
1973-01-22 15:00:00    0.226747
                         ...   
2018-09-07 23:10:00    0.133470
2018-09-07 23:20:00    0.134670
2018-09-07 23:30:00    0.141399
2018-09-07 23:40:00    0.154153
2018-09-07 23:50:00    0.143487
Length: 527882, dtype: float64
1971-01-01 10:00:00    0.122484
1971-01-01 23:00:00    0.099009
1971-01-02 07:00:00    0.136729
1971-01-08 00:00:00    0.151641
1971-01-08 01:00:00    0.232817
                         ...   
2018-09-07 23:10:00    0.177663
2018-09-07 23:20:00    0.195453
2018-09-07 23:30:00    0.211819
2018-09-07 23:40:00    0.225941
2018-09-07 23:50:00    0.236978
Length: 514820, dtype: float64
1979-01-01 11:00:00    0.110143
1979-01-03 00:00:00    0.224076
1979-01-03 01:00:00    0.186293
1979-01-03 02:00:00    0.112697
1979-01-07 10:00:00    0.131546
                         ...   
2018-09-07 17:00:00    0.099351
2018-09-07

## Save variables for future use (optional)

In [5]:
import pickle

# Variables to save
variables = [
    ("extremes_NTR", extremes_NTR),
    ("events_NTR", events_NTR),
    ("threshold_NTR", threshold_NTR),
]

# Loop and save each variable
for variable_name, variable_data in variables:
    with open(f"outputs/{variable_name}.pkl", "wb") as f:
        pickle.dump(variable_data, f)